# MSI SEAL: local seed-0 baseline

Select a local **Python 3 kernel → Run All**. The current kernel starts computation and displays results. First use creates an independent **Python 3.11 + PyTorch 2.4 CPU** environment; no kernel switch is required.

Pipeline: `MSI molecular graph + existing node2vec features → one-hop enclosing subgraphs → DRNL labels + node-type features → DGCNN → ten-fold evaluation`.

Following DREAMwalk **Supplementary Method 1**, use **128D node2vec + 4D node-type features** and validation **patience=10**. Other architecture/training defaults follow pinned SEAL/DGCNN sources. The CPU port retains their row-normalized convolution; original sources remain in `vendor/`.

Reuse the molecular graph, 5,926 positives, 5,926 saved seed-0 negatives and node2vec fold IDs. SEAL consumes node2vec features, so this is not a walk-free input pipeline. SEAL itself has no `pp` or `τ` and needs no MeSH/ATC.

New outputs go into `04.SEAL_reproduction/`. Training starts when you run its cells; published outputs are cleared.


## 1. Directories and run name

Set `REPOSITORY_OVERRIDE` to the repository root or `00.Reproduction/` only if needed. `PYTHON_OVERRIDE` may select a Python 3.11 interpreter satisfying the pins. Use a new run name after changing settings, inputs or implementation.


In [ ]:
from pathlib import Path
import ast
import csv
import html
import json
import os
import shutil
import signal
import subprocess
import sys
from IPython.display import display, HTML

RUN_NAME = 'msi_seed0'
REPOSITORY_OVERRIDE = None
PYTHON_OVERRIDE = None

def find_reproduction_root(start):
    start = Path(start).expanduser().resolve()
    for parent in [start, *start.parents]:
        for candidate in [parent/'00.Reproduction', parent]:
            if ((candidate/'00.DREAMwalk_reproduction').is_dir()
                    and (candidate/'04.SEAL_reproduction').is_dir()):
                return candidate
    raise FileNotFoundError('00.Reproduction not found; set REPOSITORY_OVERRIDE.')

REPRODUCTION = find_reproduction_root(REPOSITORY_OVERRIDE or Path.cwd())
ROOT = REPRODUCTION/'04.SEAL_reproduction'
SOURCE = REPRODUCTION/'00.DREAMwalk_reproduction'
NODE2VEC = REPRODUCTION/'01.node2vec_reproduction'
RUNNER = ROOT/'src/run_seal.py'
ENV_PREFIX = ROOT/'environment/.conda-env'
PYTHON = Path(PYTHON_OVERRIDE).expanduser().resolve() if PYTHON_OVERRIDE else ENV_PREFIX/'bin/python'
if not RUN_NAME or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_NAME):
    raise ValueError('RUN_NAME accepts only letters, digits, underscores and hyphens.')
RESULTS = ROOT/'results'/RUN_NAME
LOGS = ROOT/'logs'/RUN_NAME
LOGS.mkdir(parents=True, exist_ok=True)
required = [SOURCE/'data/processed/msi_graph.tsv', SOURCE/'data/processed/msi_nodetypes.tsv',
            SOURCE/'data/processed/msi_positive_pairs.tsv', SOURCE/'results/msi_repeat0_negative_pairs.csv',
            NODE2VEC/'data/msi_seed0/msi_node2vec_seed0.pkl',
            NODE2VEC/'results/msi_seed0/fold_assignments.csv',
            NODE2VEC/'results/msi_seed0/embedding_complete.json',
            NODE2VEC/'results/msi_seed0/run_config.json', RUNNER]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Required input missing: {path}')
print('SEAL project: ', ROOT)
print('Computation interpreter: ', PYTHON)
print('Results: ', RESULTS)
print('Reused node2vec: ', NODE2VEC/'data/msi_seed0/msi_node2vec_seed0.pkl')

## 2. Independent CPU environment and execution helpers

Python 3.11; PyTorch 2.4.0; NumPy 1.26.2; SciPy 1.11.4; scikit-learn 1.3.2. First installation requires network access. Use CPU/four threads, without CUDA, MPS or the legacy SEAL C++ extension.

This section prepares the environment only. Interrupting a computation cell terminates its subprocess group while preserving completed subgraphs, epochs and folds.


In [ ]:
def stream_command(command, log_path, env=None):
    command = list(map(str, command))
    print('Starting: ', ' '.join(command), flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\nCOMMAND: ' + repr(command) + '\n')
        with subprocess.Popen(command, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1,
                              start_new_session=True) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = process.wait()
            except BaseException:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
                raise
            if returncode:
                raise subprocess.CalledProcessError(returncode, command)

def runner_literal(name):
    tree = ast.parse(RUNNER.read_text())
    for statement in tree.body:
        if isinstance(statement, ast.Assign) and any(isinstance(t, ast.Name) and t.id == name for t in statement.targets):
            return ast.literal_eval(statement.value)
    raise KeyError(name)

PINS = runner_literal('PINS')
probe = (
    'import sys; from importlib.metadata import version; '
    'import torch, numpy, scipy, sklearn; '
    'assert sys.version_info[:2] == (3,11); '
    f'expected={PINS!r}; '
    'actual={k:version(k) for k in expected}; '
    'assert actual == expected, actual; print(sys.version); print(actual)'
)
ready = subprocess.run([str(PYTHON), '-c', probe], capture_output=True, text=True) if PYTHON.is_file() else None
if ready is None or ready.returncode:
    if PYTHON_OVERRIDE:
        if ready is not None:
            print(ready.stdout, ready.stderr)
        raise RuntimeError('PYTHON_OVERRIDE does not match package pins; clear it to create a dedicated environment.')
    candidates = [shutil.which('mamba'), shutil.which('conda'),
                  str(Path.home()/'mambaforge/bin/mamba'),
                  str(Path.home()/'miniforge3/bin/mamba'),
                  str(Path.home()/'miniforge3/bin/conda'),
                  str(Path.home()/'miniconda3/bin/conda')]
    manager = next((p for p in candidates if p and Path(p).is_file()), None)
    if manager is None:
        raise RuntimeError('mamba/conda not found; install Miniforge and rerun this section.')
    install_env = dict(os.environ)
    install_env['CONDA_PKGS_DIRS'] = str(ROOT/'environment/.conda-pkgs')
    action = 'install' if (ENV_PREFIX/'conda-meta/history').is_file() else 'create'
    stream_command([manager, action, '--yes', '--prefix', ENV_PREFIX,
                    '--override-channels', '-c', 'conda-forge', 'python=3.11', 'pip'],
                   LOGS/'environment.log', env=install_env)
    stream_command([PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check',
                    '-r', ROOT/'environment/requirements.txt'], LOGS/'environment.log')
    ready = subprocess.run([str(PYTHON), '-c', probe], capture_output=True, text=True)
    if ready.returncode:
        print(ready.stdout, ready.stderr)
        raise RuntimeError('Environment is not ready; inspect the installation log.')
print(ready.stdout)

CHILD_ENV = dict(os.environ)
CHILD_ENV['PYTHONUNBUFFERED'] = '1'
CHILD_ENV['PYTHONDONTWRITEBYTECODE'] = '1'

def run_phase(phase, fold=None):
    command = [PYTHON, '-u', RUNNER, phase, '--run-name', RUN_NAME]
    if fold is not None:
        command += ['--fold', str(fold)]
    suffix = f'_fold{fold:02d}' if fold is not None else ''
    stream_command(command, LOGS/f'{phase}{suffix}.log', env=CHILD_ENV)

def show_rows(rows):
    if not rows:
        return
    keys = list(rows[0])
    head = ''.join('<th style="text-align:left;padding:6px">'+html.escape(str(k))+'</th>' for k in keys)
    body = ''.join('<tr>'+''.join('<td style="padding:6px">'+html.escape(str(r.get(k, '')))+'</td>' for k in keys)+'</tr>' for r in rows)
    display(HTML('<table><thead><tr>'+head+'</tr></thead><tbody>'+body+'</tbody></table>'))

def show_csv(path, tail=None):
    if not path.is_file():
        print('Not generated yet: ', path)
        return
    with path.open(newline='', encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    show_rows(rows[-tail:] if tail else rows)

## 3. Fixed parameters and paper reference

| Component | Setting |
|---|---|
| Graph | Shared MSI; 29,959 nodes / 478,728 input rows; undirected simple adjacency; no DDA label edges |
| Features | Existing 128D node2vec + four node-type indicators + DRNL structural one-hot |
| Subgraphs | One hop; no per-hop node cap; remove the target edge |
| DGCNN | [32,32,32,1]; row normalization; SortPooling 60th percentile, min k=10; Conv1d 16/32; MLP 128; dropout=0.5 |
| Training | Adam; lr=1e-4; batch=50; max 50 epochs; validation-loss patience=10; CPU/four threads |
| Evaluation | Same 5,926 positives + 5,926 seed-0 negatives; saved node2vec ten-fold IDs |
| Validation | Stratified 10% of each outer training fold, seed 0; outer test data do not select epochs |
| Metrics | Accuracy, AUROC, AUPR=average precision |

**Input adapter:** reuse the audited molecular-only node2vec checkpoint. Do not call the original SEAL helper to regenerate embeddings or inject negative edges. DDA labels stay outside the graph. The complete author MSI SEAL benchmark script is unavailable; record these adapter choices explicitly.

Paper references are ten SEAL repeats from `Figure2a!A32:E41`. Local evaluation is **one seed × ten folds**, compared with SEAL's own paper baseline.


In [ ]:
CONFIG = runner_literal('CONFIG')
show_rows([{'parameter': k, 'value': v} for k, v in CONFIG.items()])
print('Paper SEAL results: ')
show_csv(ROOT/'reference/paper_seal_fig2a_summary.csv')
run_phase('config')

## 4. Prepare graph, node features and partitions

Check embedding/graph hashes, complete node coverage, pair order and saved fold IDs. Build 132D node features and save train/validation/test indices per fold. Inputs are read-only; outputs belong to this model directory.


In [ ]:
run_phase('prepare')
info = json.loads((RESULTS/'prepare_complete.json').read_text())
show_rows([{k: info[k] for k in ['nodes','molecular_rows','simple_edges','positives','negatives','node_feature_dimension','treatment_edges_added']}])

## 5. Generate one enclosing-subgraph corpus

Extract one-hop enclosing subgraphs and DRNL labels for all **11,852 pairs**. Cache them by pair ID in SQLite and read training batches from disk.

Report/save every 100 pairs and resume saved work after interruption. All folds reuse this corpus because the molecular graph contains no DDA labels and node2vec features are fixed.


In [ ]:
run_phase('subgraphs')
info = json.loads((RESULTS/'subgraphs_complete.json').read_text())
show_rows([{k: info[k] for k in ['pairs','max_nodes','label_count','sortpooling_k']}])
print('Cache: ', ROOT/'data'/RUN_NAME/'subgraphs.sqlite')

## 6. Ten-fold DGCNN training and held-out evaluation

Initialize DGCNN separately per fold. Choose the best epoch by **validation loss**, stopping after ten non-improving epochs or a maximum of 50.

Save each improved model as `best.pt`; save `last.pt`, optimizer and RNG states every epoch. Predict the test fold only with the selected best model. Reuse completed folds and resume incomplete folds from the last epoch.

The next ten cells run one fold each. Logs use `logs/<RUN_NAME>/train_foldXX.log`.


### 6.1 Fold 1/10

In [ ]:
run_phase('train', fold=1)
show_csv(RESULTS/'fold01'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold01'/'metrics.json').read_text())])

### 6.2 Fold 2/10

In [ ]:
run_phase('train', fold=2)
show_csv(RESULTS/'fold02'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold02'/'metrics.json').read_text())])

### 6.3 Fold 3/10

In [ ]:
run_phase('train', fold=3)
show_csv(RESULTS/'fold03'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold03'/'metrics.json').read_text())])

### 6.4 Fold 4/10

In [ ]:
run_phase('train', fold=4)
show_csv(RESULTS/'fold04'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold04'/'metrics.json').read_text())])

### 6.5 Fold 5/10

In [ ]:
run_phase('train', fold=5)
show_csv(RESULTS/'fold05'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold05'/'metrics.json').read_text())])

### 6.6 Fold 6/10

In [ ]:
run_phase('train', fold=6)
show_csv(RESULTS/'fold06'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold06'/'metrics.json').read_text())])

### 6.7 Fold 7/10

In [ ]:
run_phase('train', fold=7)
show_csv(RESULTS/'fold07'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold07'/'metrics.json').read_text())])

### 6.8 Fold 8/10

In [ ]:
run_phase('train', fold=8)
show_csv(RESULTS/'fold08'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold08'/'metrics.json').read_text())])

### 6.9 Fold 9/10

In [ ]:
run_phase('train', fold=9)
show_csv(RESULTS/'fold09'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold09'/'metrics.json').read_text())])

### 6.10 Fold 10/10

In [ ]:
run_phase('train', fold=10)
show_csv(RESULTS/'fold10'/'history.csv', tail=3)
show_rows([json.loads((RESULTS/'fold10'/'metrics.json').read_text())])

## 7. Summarize all ten folds

Verify exactly one held-out prediction per labeled pair, then report fold metrics and mean ± SD. Do not report a ten-fold average before every fold finishes. This SD is **between folds**, rather than the paper's between-repeat SD.


In [ ]:
run_phase('summarize')
show_csv(RESULTS/'fold_results.csv')
show_csv(RESULTS/'summary.csv')

## 8. Compare with the paper's SEAL baseline

Inspect `local_minus_paper`; positive values indicate higher local metrics. Compare all three measures with Figure 2a SEAL rows. One local repeat is not the paper's ten-repeat protocol.


In [ ]:
run_phase('compare')
show_csv(RESULTS/'paper_comparison.csv')
print('Results: ', RESULTS)
print('Each best-fold model was saved before test evaluation.')

## 9. Outputs and future controls

- Per-fold checkpoints, validation history and predictions: `results/<RUN_NAME>/foldXX/`.
- Pair/fold IDs: `fold_assignments.csv`; inner partitions: `partitions.json`.
- All held-out predictions: `oof_predictions.csv`; means: `summary.csv`; reference comparison: `paper_comparison.csv`.
- Add future SEAL controls as sections in this notebook. This baseline does not automatically add tuning or seeds.

Sources: [DREAMwalk Supplementary Method 1](https://www.nature.com/articles/s41467-023-39301-y#Sec15), [SEAL](https://github.com/muhanzhang/SEAL/tree/ca1f019a15fb0c21796042165b4e6bee73981dd3), [DGCNN](https://github.com/muhanzhang/pytorch_DGCNN/tree/50f504131ca66382f3f078ae98e59aa3ae35b795).
